# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aliakhtar1010/search-ranking-ml/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

In [1]:
import os
import json
import numpy as np
import pandas as pd

from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

RANDOM_STATE = 42

DATA_URL = (
    "https://raw.githubusercontent.com/"
    "aliakhtar1010/search-ranking-ml/main/"
    "data/raw/content_refresh_anonymized.csv"
)

df = pd.read_csv(DATA_URL)

# Same proxy target used in Weeks 5-6
df["declining_label"] = (
    df["trend_direction"].str.lower() == "down"
).astype(int)

# Exact validated feature set from Weeks 5-6
FEATURES = [
    "search_volume",
    "competition",
    "cpc",
    "word_count",
    "char_count",
    "impressions_90d",
    "clicks_90d",
    "pageviews_90d",
    "sessions_90d",
    "users_90d",
    "engaged_sessions_90d",
    "ai_sessions_90d",
    "scroll_events_90d",
    "days_with_impressions",
    "days_with_sessions",
    "content_age_days",
    "days_since_last_update",
    "ctr",
    "avg_position",
    "engagement_rate",
    "scroll_rate",
    "ai_traffic_pct"
]

X = df[FEATURES].copy()
y = df["declining_label"].copy()

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=RANDOM_STATE
)

train_idx, test_idx = next(
    splitter.split(X, y, groups=df["client_id"])
)

X_train = X.iloc[train_idx].copy()
X_test = X.iloc[test_idx].copy()

y_train = y.iloc[train_idx].copy()
y_test = y.iloc[test_idx].copy()

test_df = df.iloc[test_idx].copy()

model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(
        max_iter=2000,
        random_state=RANDOM_STATE
    ))
])

model.fit(X_train, y_train)

test_df["decline_probability"] = model.predict_proba(X_test)[:, 1]

train_clients = set(df.iloc[train_idx]["client_id"])
test_clients = set(test_df["client_id"])

print("Training rows:", len(train_idx))
print("Test rows:", len(test_idx))
print("Client overlap:", len(train_clients.intersection(test_clients)))
print("Test base rate:", round(y_test.mean(), 3))

Training rows: 23837
Test rows: 6163
Client overlap: 0
Test base rate: 0.511


In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
VISIBILITY_THRESHOLD = 731
STALE_THRESHOLD = 90
HIGH_RISK_THRESHOLD = 0.70

queue = test_df[
    [
        "content_id",
        "decline_probability",
        "impressions_90d",
        "clicks_90d",
        "ctr",
        "avg_position",
        "days_since_last_update",
        "content_age_days",
        "word_count",
        "declining_label"
    ]
].copy()

queue["is_visible"] = (
    queue["impressions_90d"] >= VISIBILITY_THRESHOLD
)

queue["is_stale"] = (
    queue["days_since_last_update"] >= STALE_THRESHOLD
)

queue["high_decline_risk"] = (
    queue["decline_probability"] >= HIGH_RISK_THRESHOLD
)

def assign_reason(row):
    if row["high_decline_risk"] and row["is_visible"] and row["is_stale"]:
        return "stale_visible_decline_risk"

    if row["high_decline_risk"] and row["is_visible"] and not row["is_stale"]:
        return "recent_visible_decline_risk"

    if row["high_decline_risk"] and not row["is_visible"]:
        return "low_visibility_decline_risk"

    return "monitor"


def assign_action(reason):
    mapping = {
        "stale_visible_decline_risk": "review_for_refresh",
        "recent_visible_decline_risk": "investigate_decline",
        "low_visibility_decline_risk": "review_if_strategic",
        "monitor": "monitor"
    }

    return mapping[reason]


queue["reason_code"] = queue.apply(assign_reason, axis=1)

queue["action_label"] = queue["reason_code"].map(assign_action)

# Model probability remains the primary ranking score
queue["action_score"] = queue["decline_probability"]

queue = queue.sort_values(
    "action_score",
    ascending=False
).reset_index(drop=True)

queue["rank"] = np.arange(1, len(queue) + 1)

print("Queue rows:", len(queue))
print("\nReason-code counts:")
print(queue["reason_code"].value_counts())

display(
    queue[
        [
            "rank",
            "content_id",
            "action_score",
            "impressions_90d",
            "days_since_last_update",
            "reason_code",
            "action_label"
        ]
    ].head(20)
)

Queue rows: 6163

Reason-code counts:
reason_code
monitor                        5345
recent_visible_decline_risk     415
low_visibility_decline_risk     301
stale_visible_decline_risk      102
Name: count, dtype: int64


,rank,content_id,action_score,impressions_90d,days_since_last_update,reason_code,action_label
0,1,content_b08562686d22,0.890543,2846,106,stale_visible_decline_risk,review_for_refresh
1,2,content_8ede62882d0b,0.884867,556,20,low_visibility_decline_risk,review_if_strategic
2,3,content_453722754fea,0.876679,140079,20,recent_visible_decline_risk,investigate_decline
3,4,content_374e795aab68,0.866905,235,20,low_visibility_decline_risk,review_if_strategic
4,5,content_c94a53e3bfb8,0.865646,2164,20,recent_visible_decline_risk,investigate_decline
5,6,content_26d48a980581,0.864896,1266,106,stale_visible_decline_risk,review_for_refresh
6,7,content_a928cb66d230,0.862645,128,20,low_visibility_decline_risk,review_if_strategic
7,8,content_87c007fb5c26,0.859521,2463,20,recent_visible_decline_risk,investigate_decline
8,9,content_b51992d6cbc2,0.858300,1578,20,recent_visible_decline_risk,investigate_decline
9,10,content_c84a0ab98e90,0.858024,223271,20,recent_visible_decline_risk,investigate_decline


### Ranked Actions

The validated Logistic Regression model produces a probability that each held-out webpage belongs to the decline-proxy class. I use this probability as the primary ranking score.

The score itself is not the action. I translate the ranked output into human-readable review archetypes using information that was already available to the model:

- **stale_visible_decline_risk** → high predicted decline risk, meaningful existing visibility, and at least 90 days since the last update. Review for a possible refresh first.
- **recent_visible_decline_risk** → high predicted decline risk and meaningful visibility, but the page was updated relatively recently. Investigate ranking, demand, intent, or competition before assuming freshness is the issue.
- **low_visibility_decline_risk** → high predicted decline risk but little existing visibility. Review only when strategically important because the immediate recoverable value may be lower.
- **monitor** → lower predicted decline risk. No immediate refresh recommendation from this system.

These are decision-support labels rather than claims that refreshing a page will improve performance.

## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

### Decay / Refresh Insight

The earlier signal analysis and FlyRank research suggest that freshness can be a useful review signal, but the validated model shows that staleness alone is not enough to drive the queue. Only a small share of the highest-ranked pages are stale by the 90-day threshold.

I therefore treat staleness as one review clue rather than a causal trigger. Older, visible, high-risk pages receive a refresh-oriented reason code, while recently updated high-risk pages are routed to broader decline investigation instead of assuming another refresh is the correct action.

### Intended Use

This playbook is intended for an SEO or content team with more existing webpages than it can manually inspect.

The model's role is to narrow that inventory into a ranked review queue. A human can begin with the highest-ranked pages, inspect the supplied reason code and supporting search signals, and decide whether the correct next step is a refresh, further investigation, monitoring, or no action.

### Cost / Value Thinking

Human review and content editing have real costs. For that reason, the goal is not to flag as many pages as possible. Precision near the top of the queue matters more than generating a very large list.

The validated client-grouped model achieved Precision@20 of 0.75 and Precision@50 of 0.72 in the previous analysis. These values support using the top of the queue as a decision-support shortlist, but they do not imply that every recommendation is correct.

Existing search visibility also provides useful value context. A high-risk page with substantial impressions may deserve review sooner than a similarly risky page with almost no measured visibility because there is more existing search exposure to investigate.

### Limits

This system does **not** establish that:

- content staleness causes decline;
- every declining page requires a refresh;
- refreshing a recommended page will improve traffic;
- model probability represents guaranteed future performance;
- low-ranked pages are healthy;
- the model should replace editorial or SEO judgment.

The decline label is a proxy for review priority, not direct ground truth for whether a refresh will create value.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
top20 = queue.head(20)
top50 = queue.head(50)

summary = pd.DataFrame([
    {
        "queue_segment": "Top 20",
        "pages": len(top20),
        "observed_decline_rate": top20["declining_label"].mean(),
        "median_impressions": top20["impressions_90d"].median(),
        "stale_pages": int(top20["is_stale"].sum())
    },
    {
        "queue_segment": "Top 50",
        "pages": len(top50),
        "observed_decline_rate": top50["declining_label"].mean(),
        "median_impressions": top50["impressions_90d"].median(),
        "stale_pages": int(top50["is_stale"].sum())
    }
])

summary

,queue_segment,pages,observed_decline_rate,median_impressions,stale_pages
0,Top 20,20,0.75,1652.0,3
1,Top 50,50,0.72,1255.0,8


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

### Human Review Rules

Before acting on a recommendation, a reviewer should check:

1. **Search demand:** Has interest in the topic fallen independently of the page?
2. **SERP / competition:** Did competing pages or search-result features change?
3. **Content accuracy:** Is the page genuinely outdated, incomplete, or incorrect?
4. **Search intent:** Does the page still match what users appear to want?
5. **Recent changes:** Was the page recently updated, migrated, redirected, or intentionally repositioned?
6. **Seasonality:** Could the measured decline be expected for this topic or time period?
7. **Business importance:** Is the page strategically valuable enough to justify editing effort?
8. **Cannibalization / overlap:** Could another page be competing for the same demand?

### No-Go List — What Should Not Be Automated

The model should **not automatically**:

- rewrite or publish webpage content;
- delete, redirect, or unpublish pages;
- change titles or metadata;
- merge pages;
- infer that the decline was caused by stale content;
- promise traffic recovery;
- take action solely because a probability threshold was crossed.

The safest workflow is:

**model ranks → human investigates → human chooses action → performance is monitored afterward.**

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
review_rules = pd.DataFrame([
    ["search_demand", "Check whether topic demand changed independently of the page."],
    ["competition", "Inspect SERP and competitor changes."],
    ["content_accuracy", "Confirm whether content is actually outdated or incomplete."],
    ["search_intent", "Check whether the page still satisfies current intent."],
    ["recent_changes", "Check recent edits, migrations, redirects, or repositioning."],
    ["seasonality", "Check whether decline is expected for the topic or period."],
    ["business_value", "Confirm the page is worth the cost of intervention."],
    ["cannibalization", "Check for overlapping internal pages or demand."]
], columns=["review_check", "reviewer_question"])

review_rules

,review_check,reviewer_question
0,search_demand,Check whether topic demand changed independent...
1,competition,Inspect SERP and competitor changes.
2,content_accuracy,Confirm whether content is actually outdated o...
3,search_intent,Check whether the page still satisfies current...
4,recent_changes,"Check recent edits, migrations, redirects, or ..."
5,seasonality,Check whether decline is expected for the topi...
6,business_value,Confirm the page is worth the cost of interven...
7,cannibalization,Check for overlapping internal pages or demand.


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

### Monitoring and Retraining

The model should not be retrained merely because time has passed. Retraining should be triggered by evidence that the data or ranking quality has changed.

I would monitor:

- **Precision@20 and Precision@50:** investigate if ranking quality materially declines on newly reviewed batches.
- **Decline base rate:** a substantial shift means the target population may have changed.
- **Feature distributions:** large changes in traffic, age, engagement, or visibility characteristics can indicate data drift.
- **New client behavior:** performance should be checked when substantially different client populations enter the system.
- **Schema / measurement changes:** changes to GSC, GA4, feature definitions, or missing-data behavior require revalidation.
- **Reason-code mix:** a major shift in which archetypes dominate the queue may indicate changing population characteristics.

Any retrained model should repeat the leakage audit and client-grouped validation before replacing the previous model.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
monitoring_triggers = pd.DataFrame([
    {
        "signal": "Precision@50",
        "current_reference": 0.72,
        "trigger": "Material sustained decline on reviewed future batches",
        "response": "Audit errors and revalidate model"
    },
    {
        "signal": "Precision@20",
        "current_reference": 0.75,
        "trigger": "Material sustained decline on reviewed future batches",
        "response": "Audit top-ranked recommendations"
    },
    {
        "signal": "Decline base rate",
        "current_reference": round(float(y_test.mean()), 3),
        "trigger": "Large population shift",
        "response": "Recheck thresholding and validation"
    },
    {
        "signal": "Feature distributions",
        "current_reference": "Week-7 validated population",
        "trigger": "Large distribution shift",
        "response": "Investigate data drift"
    },
    {
        "signal": "Schema / data availability",
        "current_reference": "Current 22-feature contract",
        "trigger": "Column definition or availability changes",
        "response": "Rebuild data contract and revalidate"
    }
])

monitoring_triggers

,signal,current_reference,trigger,response
0,Precision@50,0.72,Material sustained decline on reviewed future ...,Audit errors and revalidate model
1,Precision@20,0.75,Material sustained decline on reviewed future ...,Audit top-ranked recommendations
2,Decline base rate,0.511,Large population shift,Recheck thresholding and validation
3,Feature distributions,Week-7 validated population,Large distribution shift,Investigate data drift
4,Schema / data availability,Current 22-feature contract,Column definition or availability changes,Rebuild data contract and revalidate


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

### Paper Exports

The ranked queue is exported so the final research paper can reuse the exact recommendations generated by this notebook.

The CSV contains only pseudonymized content identifiers and public-safe analytical fields. It does not contain client names, URLs, private queries, or raw private content.

The queue CSV remains outside Git by design and can be regenerated by running this notebook. A compact metrics JSON is also written as a reproducible receipt for the headline playbook numbers.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
os.makedirs("work/outputs", exist_ok=True)

QUEUE_PATH = "work/outputs/action_playbook_queue.csv"
METRICS_PATH = "work/outputs/action_playbook_metrics.json"

export_columns = [
    "rank",
    "content_id",
    "action_score",
    "impressions_90d",
    "clicks_90d",
    "ctr",
    "avg_position",
    "days_since_last_update",
    "content_age_days",
    "reason_code",
    "action_label"
]

queue[export_columns].to_csv(
    QUEUE_PATH,
    index=False
)

playbook_metrics = {
    "random_state": RANDOM_STATE,
    "test_pages": int(len(queue)),
    "test_clients": int(len(test_clients)),
    "client_overlap": int(
        len(train_clients.intersection(test_clients))
    ),
    "test_base_rate": float(y_test.mean()),

    # validated Week-6 receipts
    "validated_precision_at_20": 0.75,
    "validated_precision_at_50": 0.72,

    "top20_observed_decline_rate": float(
        queue.head(20)["declining_label"].mean()
    ),
    "top50_observed_decline_rate": float(
        queue.head(50)["declining_label"].mean()
    ),

    "high_risk_pages": int(
        queue["high_decline_risk"].sum()
    ),

    "stale_visible_decline_risk": int(
        (queue["reason_code"] == "stale_visible_decline_risk").sum()
    ),

    "recent_visible_decline_risk": int(
        (queue["reason_code"] == "recent_visible_decline_risk").sum()
    ),

    "low_visibility_decline_risk": int(
        (queue["reason_code"] == "low_visibility_decline_risk").sum()
    )
}

with open(METRICS_PATH, "w") as f:
    json.dump(playbook_metrics, f, indent=2)

print("Saved:", QUEUE_PATH)
print("Saved:", METRICS_PATH)

print("\nMetrics receipt:")
print(json.dumps(playbook_metrics, indent=2))

Saved: work/outputs/action_playbook_queue.csv
Saved: work/outputs/action_playbook_metrics.json

Metrics receipt:
{
  "random_state": 42,
  "test_pages": 6163,
  "test_clients": 7,
  "client_overlap": 0,
  "test_base_rate": 0.5109524582184002,
  "validated_precision_at_20": 0.75,
  "validated_precision_at_50": 0.72,
  "top20_observed_decline_rate": 0.75,
  "top50_observed_decline_rate": 0.72,
  "high_risk_pages": 818,
  "stale_visible_decline_risk": 102,
  "recent_visible_decline_risk": 415,
  "low_visibility_decline_risk": 301
}


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.